In [ ]:
# CPT_PROCEDURES.ipynb

# ======================
# Step 1. Import libraries
# ======================
import pandas as pd
import os

base_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_path = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

# ======================
# Step 2. Load source data
# ======================
hcpcs_events = pd.read_csv(os.path.join(base_path, "hosp_hcpcsevents.csv"))
hcpcs_dict = pd.read_csv(os.path.join(base_path, "hosp_d_hcpcs.csv"))

print("hosp_hcpcsevents:", hcpcs_events.shape)
print("hosp_d_hcpcs:", hcpcs_dict.shape)

# ======================
# Step 3. Preprocess dictionary table
# ======================
hcpcs_dict = hcpcs_dict.rename(columns={
    "code": "hcpcs_cd",
    "short_description": "dict_short_description",
    "long_description": "dict_long_description"
})

# ======================
# Step 4. Merge events with dictionary
# ======================
hcpcs_merged = hcpcs_events.merge(
    hcpcs_dict[["hcpcs_cd", "dict_short_description", "dict_long_description"]],
    on="hcpcs_cd",
    how="left"
)

print("Merged events + dictionary:", hcpcs_merged.shape)

# ======================
# Step 5. Build final CPT_PROCEDURES table
# ======================
cpt_final = pd.DataFrame({
    "pat_id": hcpcs_merged["subject_id"],
    "csn": hcpcs_merged["hadm_id"],
    "procedure_cpt_code": hcpcs_merged["hcpcs_cd"],
    "procedure_cpt_desc": hcpcs_merged["dict_long_description"].fillna(hcpcs_merged["dict_short_description"]),
    "procedure_dttm": hcpcs_merged["chartdate"]
})

print("✅ Final CPT_PROCEDURES:", cpt_final.shape)
display(cpt_final.head(10))

# ======================
# Step 6. Save output
# ======================
out_file = os.path.join(output_path, "CPT_PROCEDURES.csv")
cpt_final.to_csv(out_file, index=False)
print(f"CPT_PROCEDURES file saved to {out_file}")


hosp_hcpcsevents: (186074, 6)
hosp_d_hcpcs: (89208, 4)
Merged events + dictionary: (186074, 8)
✅ Final CPT_PROCEDURES: (186074, 5)


,pat_id,csn,procedure_cpt_code,procedure_cpt_desc,procedure_dttm
0,10000068,25022803,99218,Hospital observation services,2160-03-04
1,10000084,29888819,G0378,"Hospital observation service, per hour",2160-12-28
2,10000108,27250926,99219,Hospital observation services,2163-09-27
3,10000117,22927623,43239,Digestive system,2181-11-15
4,10000117,22927623,G0378,"Hospital observation service, per hour",2181-11-15
5,10000161,22148160,99219,Hospital observation services,2163-08-20
6,10000248,20600184,99219,Hospital observation services,2192-11-30
7,10000280,25852320,99219,Hospital observation services,2151-03-18
8,10000635,20642640,G0378,"Hospital observation service, per hour",2143-12-23
9,10000635,26134563,G0378,"Hospital observation service, per hour",2136-06-19


CPT_PROCEDURES file saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/CPT_PROCEDURES.csv
